# Null model comparison

Three things shown here:
1. **Why the grand-mean null biases R²** — synthetic demonstration
2. **Empirical R² distributions** from the training-mean null (layer 18)
3. **Why PTYFS is NaN** and how deviance R² fixes it

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pickle, os

rng = np.random.default_rng(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## Part 1 — Why grand-mean null biases R²

Simulate a neuron whose firing rate drifts upward over time (typical neural non-stationarity).
The grand-mean null uses the global mean (including test data), so it tracks the test distribution better than the training-mean null does — giving it an *unfair advantage* as a null.

Concretely, under the grand-mean null the apparent R² of a **noise** model is systematically negative, meaning the model has to beat a biased baseline.

In [ ]:
# --- Synthetic neuron: Poisson with a linear upward drift ---
T          = 2000          # total words / time-bins
drift_rate = 0.003         # spikes / word / bin (slow upward drift)
base_rate  = 0.05          # starting rate
true_rate  = base_rate + drift_rate * np.arange(T)
Y_syn      = rng.poisson(true_rate).astype(float)  # simulated spike counts

# 5-fold time-block CV split
K = 5
block = T // K
folds = [(np.r_[np.arange(0, i*block), np.arange((i+1)*block, T)],
          np.arange(i*block, (i+1)*block))
         for i in range(K)]

# --- Compute per-fold null LL under both null choices ---
grand_mean  = Y_syn.mean()                            # uses ALL data

ll_null_grand = 0.0
ll_null_train = 0.0

for tr_idx, te_idx in folds:
    Y_tr, Y_te  = Y_syn[tr_idx], Y_syn[te_idx]
    mu_tr       = Y_tr.mean().clip(1e-10)             # training mean only
    n_te        = len(te_idx)

    ll_null_grand += (Y_te * np.log(grand_mean) - grand_mean).sum()
    ll_null_train += (Y_te * np.log(mu_tr)      - mu_tr).sum()

print(f"Grand-mean null  LL: {ll_null_grand:+.1f}")
print(f"Training-mean null LL: {ll_null_train:+.1f}")
print()
print("A 'random-features' model should give R² ≈ 0.")
print("With grand-mean null, ll_model ≈ ll_null_train (can't beat intercept fit")
print(" on training), so R²_grand = 1 - ll_model/ll_null_grand < 0 (biased low).")

In [ ]:
# Sweep over different drift rates and show R² bias
drift_rates = np.linspace(0, 0.010, 30)
n_neurons   = 200

bias_grand  = []   # median R² for noise model, grand-mean null
bias_train  = []   # median R² for noise model, training-mean null

for dr in drift_rates:
    r2_g_all, r2_t_all = [], []
    for _ in range(n_neurons):
        tr_ = base_rate + dr * np.arange(T)
        y_  = rng.poisson(tr_).astype(float)
        # Noise-model: predict training mean for each fold (same as null)
        # ll_model uses random features → approximate with training-mean pred
        ll_m, ll_ng, ll_nt = 0.0, 0.0, 0.0
        gm   = y_.mean()
        for tri, tei in folds:
            y_te = y_[tei]
            mu_t = y_[tri].mean().clip(1e-10)
            # Noise model = predict (slightly noisy) training mean
            mu_noise = mu_t * np.exp(rng.normal(0, 0.05))
            ll_m  += (y_te * np.log(mu_noise.clip(1e-10)) - mu_noise).sum()
            ll_ng += (y_te * np.log(gm) - gm).sum()
            ll_nt += (y_te * np.log(mu_t) - mu_t).sum()
        r2_g_all.append(1 - ll_m / ll_ng  if abs(ll_ng) > 0.1 else np.nan)
        r2_t_all.append(1 - ll_m / ll_nt  if abs(ll_nt) > 0.1 else np.nan)
    bias_grand.append(np.nanmedian(r2_g_all))
    bias_train.append(np.nanmedian(r2_t_all))

fig, ax = plt.subplots(figsize=(7, 4))
ax.axhline(0, color='k', lw=0.8, ls='--', label='unbiased (R²=0)')
ax.plot(drift_rates * 1000, bias_grand, 'C1-o', ms=4, label='grand-mean null')
ax.plot(drift_rates * 1000, bias_train, 'C0-s', ms=4, label='training-mean null (ours)')
ax.set_xlabel('Temporal drift  (Δspikes/word × 10³)')
ax.set_ylabel('Median R²  (noise model)')
ax.set_title('Null bias: R² of a noise model should be 0')
ax.legend()
ax.set_ylim(-0.5, 0.15)
plt.tight_layout()
plt.savefig('../figures/null_bias_simulation.png', bbox_inches='tight')
plt.show()
print("Grand-mean null gives increasingly negative R² as drift grows.")
print("Training-mean null stays near 0 regardless of drift.")

## Part 2 — Empirical R² distribution at layer 18

In [ ]:
DATA_DIR = '/scratch/aniluchavez/ConvoDATAS/SemanticGLM'

with open(os.path.join(DATA_DIR, 'L18_all.pkl'), 'rb') as f:
    df = pickle.load(f)

print(df.columns.tolist())
print(df.shape)
df.head(3)

In [ ]:
# Summary table
summary = []
for (reg, cond), g in df.groupby(['region', 'condition']):
    n_tot = len(g)
    n_sig = g['significant'].sum()
    med_r2 = g['r2'].median()
    med_r2_sig = g.loc[g['significant'], 'r2'].median() if n_sig else np.nan
    summary.append(dict(region=reg, condition=cond,
                        n_total=n_tot, n_sig=n_sig,
                        pct_sig=100*n_sig/n_tot,
                        median_r2=med_r2,
                        median_r2_sig=med_r2_sig))
pd.DataFrame(summary).round(4)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharey=False)
combos = [('hippocampus','self'), ('hippocampus','other'),
          ('ACC','self'), ('ACC','other')]

for ax, (reg, cond) in zip(axes.flat, combos):
    sub = df[(df['region']==reg) & (df['condition']==cond)]['r2'].dropna()
    sig = df[(df['region']==reg) & (df['condition']==cond) & df['significant']]['r2'].dropna()

    if sub.empty:
        ax.set_visible(False)
        continue

    # Clip for display
    clipped = sub.clip(-0.5, 0.5)
    ax.hist(clipped, bins=60, color='steelblue', alpha=0.7, label=f'all  n={len(sub)}')
    if len(sig):
        ax.hist(sig.clip(-0.5, 0.5), bins=30, color='C3', alpha=0.8,
                label=f'sig  n={len(sig)}')
    ax.axvline(0, color='k', lw=1, ls='--')
    ax.set_title(f'{reg} / {cond}')
    ax.set_xlabel('R² (training-mean null)')
    ax.set_ylabel('# neurons')
    ax.legend(fontsize=9)
    # Annotate median
    ax.axvline(sub.median(), color='navy', lw=1.5, ls=':',
               label=f'median={sub.median():.4f}')
    ax.legend(fontsize=9)

fig.suptitle('Layer 18  —  R² distribution  (training-mean null, block-shuffle perm test)', y=1.01)
plt.tight_layout()
plt.savefig('../figures/L18_r2_distribution.png', bbox_inches='tight')
plt.show()

## Part 3 — PTYFS NaN: the ll_null > 0 regime

For Poisson log-likelihood, `ll_null = Σ_t (y_t·log(μ) - μ)`.

Setting `y_t ≈ μ` (neuron at steady state): `ll_null ≈ n·μ·(log μ − 1)`, which is **positive** when `μ > e ≈ 2.72` spikes / bin.

PTYFS neurons fire at 5.6–14.3 spikes/word — all above the threshold.
The formula `R² = 1 − ll_model / ll_null` only works when `ll_null < 0`.

The deviance R²: `R²_dev = (ll_model − ll_null) / (ll_sat − ll_null)` is always valid.

* denominator = D_null / 2 ≥ 0 always (saturated LL ≥ any model)
* invariant to row-shuffle of Y_te (sum of `y·log y` doesn't change under permutation)
* R²_dev = 0 iff model = intercept-only; R²_dev = 1 iff perfect prediction

In [ ]:
# Visualize: ll_null(μ) vs μ  — shows when null becomes positive
mu_vals = np.linspace(0.1, 15, 500)
ll_null_per_bin = mu_vals * (np.log(mu_vals) - 1)   # n=1, y≈μ approximation

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(mu_vals, ll_null_per_bin, 'C0', lw=2)
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.axvline(np.e, color='C3', lw=1.5, ls=':', label=f'μ = e ≈ {np.e:.2f}')
ax.fill_between(mu_vals, ll_null_per_bin, 0,
                where=ll_null_per_bin < 0, alpha=0.15, color='C0', label='ll_null < 0  (formula valid)')
ax.fill_between(mu_vals, ll_null_per_bin, 0,
                where=ll_null_per_bin > 0, alpha=0.2, color='C3', label='ll_null > 0  (formula breaks → NaN)')
# PTYFS range
ax.axvspan(5.6, 14.3, alpha=0.12, color='gold', label='PTYFS range (5.6–14.3 sp/word)')
ax.set_xlabel('Mean spike rate  (spikes / word)')
ax.set_ylabel('ll_null per word  (y ≈ μ approximation)')
ax.set_title('Training-mean null LL as a function of firing rate')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../figures/llnull_regime.png', bbox_inches='tight')
plt.show()

In [ ]:
# Show how deviance R² fixes the problem (simulation)
# Simulate a neuron at each rate: noise model vs signal model
T2 = 3000
K2 = 5
b2 = T2 // K2
folds2 = [(np.r_[np.arange(0, i*b2), np.arange((i+1)*b2, T2)],
           np.arange(i*b2, (i+1)*b2)) for i in range(K2)]

mu_test_vals = [0.1, 0.5, 1.0, 2.0, 3.0, 5.0, 8.0, 12.0]
results = []

for mu in mu_test_vals:
    y = rng.poisson(mu, size=T2).astype(float)
    ll_m, ll_ng_old, ll_sat_t, ll_nt = 0., 0., 0., 0.
    for tri, tei in folds2:
        y_te = y[tei]
        mu_t = y[tri].mean().clip(1e-10)
        # Noise model = training mean + tiny noise
        mu_n = mu_t * np.exp(rng.normal(0, 0.03))
        ll_m   += (y_te * np.log(mu_n.clip(1e-10)) - mu_n).sum()
        ll_nt  += (y_te * np.log(mu_t) - mu_t).sum()
        ll_sat_t += (y_te * np.log(y_te.clip(1e-10)) - y_te).sum()

    r2_old = (1 - ll_m / ll_nt) if abs(ll_nt) > 0.1 else np.nan
    d_null = ll_sat_t - ll_nt
    r2_dev = ((ll_m - ll_nt) / d_null) if d_null > 0.1 else np.nan
    results.append(dict(mu=mu, r2_old=r2_old, r2_dev=r2_dev,
                        ll_null=ll_nt, ll_sat=ll_sat_t))

res_df = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(res_df.mu, res_df.r2_old, 'C1-o', ms=6, label='R² = 1 − ll_m/ll_null  (breaks at μ>e)')
ax.plot(res_df.mu, res_df.r2_dev, 'C0-s', ms=6, label='Deviance R² = (ll_m−ll_null)/(ll_sat−ll_null)')
ax.axhline(0, color='k', lw=0.8, ls='--', label='expected = 0 (noise model)')
ax.axvline(np.e, color='gray', lw=1, ls=':')
ax.set_xlabel('True firing rate  (spikes / word)')
ax.set_ylabel('R² of noise model  (should be ≈ 0)')
ax.set_title('Deviance R² works across all firing rates')
ax.legend(fontsize=9)
ax.set_ylim(-0.2, 0.15)
plt.tight_layout()
plt.savefig('../figures/deviance_r2_vs_old.png', bbox_inches='tight')
plt.show()

print("Old formula gives NaN (or wrong values) above μ=e; deviance R² stays near 0.")